# Multi-PMT Optical Modules in Prometheus

Prometheus supports two classes of optical module:

- **Legacy (IceCube-style):** A single spherical DOM with angular sensitivity read from `as.dat`. This is the default.
- **Next-gen:** A module with an explicit shape (spheroid or cylinder), multiple PMTs, and per-type configuration written to `om.conf`/`om.map`. Examples include the IceCube-Upgrade DEgg and the WOM.

This notebook walks through building next-gen modules, explaining every parameter, showing the configuration files they produce, running a photon propagation, and discussing how to modify the geometry and acceptance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from prometheus.detector.module import Module
from prometheus.detector.detector import Detector
from prometheus.detector.medium import Medium

## 1. The `Module` class

Every optical module is represented by a `Module` object. The parameters split into three groups.

### 1.1 Bookkeeping parameters

| Parameter | Type | Default | Meaning |
|-----------|------|---------|---------|
| `pos` | `np.ndarray` | required | Position in the detector frame [m] |
| `key` | `(int, int)` | required | `(string_id, om_id)` — **om_id must be ≥ 1** (PPC rejects om_id = 0) |
| `noise_rate` | float | `1e3` | Dark-noise rate [GHz] |
| `efficiency` | float | `0.2` | Quantum efficiency scale factor |
| `serial_no` | str or None | `None` | Hardware serial number written to the geo-f2k file; a random placeholder is used when `None` |

### 1.2 Shape parameters

These describe the physical envelope of the module, which PPC uses to place photon-hit points on the surface.

| Parameter | Type | Default | Meaning |
|-----------|------|---------|---------|
| `Rr` | float | `0.16510` m | Horizontal (equatorial) semi-axis [m] |
| `Rz` | float | `0.16510` m | Vertical semi-axis [m]. **Negative → cylindrical** with radius `Rr` and half-height `|Rz|` |
| `beta` | float | `0.49` | Angular sensitivity shape exponent (see §2) |

When `Rr == Rz > 0` the shape is a **sphere** (standard IceCube DOM).  
When `Rr != Rz > 0` the shape is a **prolate or oblate spheroid** (e.g. DEgg).  
When `Rz < 0` the shape is a **cylinder** with radius `Rr` and half-height `|Rz|` (e.g. WOM/mDOM tube).

In [ ]:
# Visualise the three shape families
fig, axes = plt.subplots(1, 3, figsize=(12, 4), subplot_kw={"aspect": "equal"})
theta = np.linspace(0, 2 * np.pi, 300)

# Sphere  (IceCube DOM)
R = 0.16510
axes[0].plot(R * np.cos(theta), R * np.sin(theta), lw=2)
axes[0].set_title(f"Sphere\nRr = Rz = {R} m")

# Prolate spheroid  (DEgg)
Rr, Rz = 0.150, 0.267
axes[1].plot(Rr * np.cos(theta), Rz * np.sin(theta), lw=2, color="tab:orange")
axes[1].set_title(f"Spheroid (DEgg)\nRr = {Rr} m, Rz = {Rz} m")

# Cylinder  (WOM)
Rr_c, Rz_c = 0.06, 0.38
rect_x = [-Rr_c, Rr_c, Rr_c, -Rr_c, -Rr_c]
rect_y = [-Rz_c, -Rz_c, Rz_c, Rz_c, -Rz_c]
axes[2].plot(rect_x, rect_y, lw=2, color="tab:green")
axes[2].set_title(f"Cylinder (WOM)\nRr = {Rr_c} m, |Rz| = {Rz_c} m")

for ax in axes:
    ax.set_xlabel("x [m]")
    ax.set_ylabel("z [m]")
    ax.grid(True, alpha=0.3)

fig.suptitle("Module shape families (cross-section)", y=1.02)
plt.tight_layout()
plt.show()

### 1.3 Next-gen / multi-PMT parameters

These parameters are only used when `module_type != -1`, which triggers PPC's *nextgen* mode.

| Parameter | Type | Default | Meaning |
|-----------|------|---------|---------|
| `module_type` | int | `-1` | PPC type ID. `-1` = legacy (uses `as.dat`). Any other integer triggers nextgen mode and becomes an entry in `om.conf` |
| `area` | float | `1.0` | Overall photon-detection efficiency scale written to `om.conf` |
| `n_pmts` | int | `1` | Number of PMTs in this module |
| `pmt_dirs` | list of `(zenith_deg, azimuth_deg)` | `[(180.0, 0.0)]` | Pointing direction of each PMT. Length must equal `n_pmts` |
| `cable_azimuth` | float or None | `None` | Azimuth of the cable attachment point [deg], written to `om.conf` if provided |

**PMT direction convention:** angles are in degrees, measured in the module's local frame.  
`(180, 0)` = pointing straight down (toward ice below).  
`(0, 0)` = pointing straight up.

## 2. Angular acceptance: the `beta` parameter

PPC models the angular sensitivity of a PMT as

$$
A(\cos\theta) = \frac{1 + \beta \cos\theta}{1 + \beta}
$$

where $\theta$ is the angle between the photon direction and the PMT normal.

- `beta = 0` → isotropic (flat response in $\cos\theta$)
- `beta > 0` → forward-peaked (photons arriving head-on are favoured)
- `beta < 0` → backward-peaked (unusual; useful for cylindrical side-collecting PMTs)
- The IceCube default is `beta = 0.49`

This is written directly into `om.conf` and PPC applies it independently for each PMT in the module.

In [ ]:
cos_theta = np.linspace(-1, 1, 300)

fig, ax = plt.subplots(figsize=(7, 4))
for beta, label in [(0.0, "beta=0 (isotropic)"),
                    (0.49, "beta=0.49 (IceCube default)"),
                    (2.0, "beta=2.0 (strongly forward)"),
                    (-2.0, "beta=-2.0 (backward, e.g. cylinder)")]:
    A = (1 + beta * cos_theta) / (1 + beta)
    ax.plot(cos_theta, A, label=label)

ax.axvline(0, color="k", lw=0.5, ls="--")
ax.set_xlabel(r"$\cos\theta$ (1 = head-on)")
ax.set_ylabel("Relative acceptance")
ax.set_title("PMT angular acceptance vs beta")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Building modules

### 3.1 Legacy IceCube DOM (baseline)

No `module_type` needed — everything defaults to the spherical, single-PMT legacy behaviour.

In [ ]:
legacy_dom = Module(
    pos=np.array([0.0, 0.0, 0.0]),
    key=(1, 1),
    # All defaults: Rr=Rz=0.16510 m, beta=0.49, n_pmts=1, module_type=-1
)
print("module_type:", legacy_dom.module_type)   # -1 → legacy
print("n_pmts    :", legacy_dom.n_pmts)
print("pmt_dirs  :", legacy_dom.pmt_dirs)        # single downward PMT
print("Rr, Rz    :", legacy_dom.Rr, legacy_dom.Rz)

### 3.2 DEgg — prolate spheroid, 2 PMTs

The DEgg has one upward-facing PMT and one downward-facing PMT, inside a prolate spheroid shell (taller than wide).

In [ ]:
degg = Module(
    pos=np.array([0.0, 0.0, 0.0]),
    key=(1, 1),
    module_type=120,      # Arbitrary integer ID; must match what you put in om.wv_120.0
    Rr=0.150,             # Equatorial radius [m]
    Rz=0.267,             # Polar semi-axis [m], Rz > Rr → prolate
    beta=0.5,             # Slightly forward-peaked acceptance
    area=1.0,             # No global efficiency re-scaling
    n_pmts=2,
    pmt_dirs=[
        (180.0, 0.0),     # PMT 0: pointing down
        (0.0,   0.0),     # PMT 1: pointing up
    ],
)
print("module_type:", degg.module_type)
print("n_pmts    :", degg.n_pmts)
print("pmt_dirs  :", degg.pmt_dirs)
print("Rr, Rz    :", degg.Rr, degg.Rz, "→ aspect ratio Rz/Rr =", round(degg.Rz / degg.Rr, 3))

### 3.3 WOM — cylinder, 2 PMTs

A cylindrical module is indicated by `Rz < 0`. PPC interprets this as radius `Rr` and half-height `|Rz|`. The negative beta reflects that the WOM PMTs collect light hitting the side, so backward-relative-to-axis is preferred.

In [ ]:
wom = Module(
    pos=np.array([0.0, 0.0, 10.0]),
    key=(1, 2),
    module_type=200,      # Different type ID from DEgg
    Rr=0.06,              # Cylinder radius [m]
    Rz=-0.38,             # Negative → cylinder; half-height = 0.38 m
    beta=-2.0,            # Side-collecting: backward-peaked
    area=1.0,
    n_pmts=2,
    pmt_dirs=[
        (180.0, 0.0),
        (0.0,   0.0),
    ],
)
print("Cylindrical?", wom.Rz < 0)
print("Radius:", wom.Rr, "m, half-height:", abs(wom.Rz), "m")

### 3.4 mDOM-style — sphere, many PMTs

A module with more than 2 PMTs pointing in arbitrary directions. The `pmt_dirs` list can be any length as long as it matches `n_pmts`.

In [ ]:
# 6-PMT module pointing along the faces of a cube
mdom_dirs = [
    (0.0,   0.0),    # up
    (180.0, 0.0),    # down
    (90.0,  0.0),    # north
    (90.0, 90.0),    # east
    (90.0, 180.0),   # south
    (90.0, 270.0),   # west
]

mdom = Module(
    pos=np.array([0.0, 0.0, 0.0]),
    key=(1, 3),
    module_type=300,
    Rr=0.16510,
    Rz=0.16510,      # Sphere
    beta=0.49,
    n_pmts=6,
    pmt_dirs=mdom_dirs,
)
print(f"6-PMT module — n_pmts={mdom.n_pmts}, directions:")
for i, (zen, az) in enumerate(mdom.pmt_dirs):
    print(f"  PMT {i}: zenith={zen}°, azimuth={az}°")

### 3.5 Validation: mismatched `pmt_dirs` raises immediately

In [ ]:
try:
    bad = Module(
        pos=np.zeros(3),
        key=(1, 99),
        module_type=400,
        n_pmts=3,
        pmt_dirs=[(0.0, 0.0), (180.0, 0.0)],   # only 2 directions for 3 PMTs
    )
except ValueError as e:
    print("Caught:", e)

## 4. Building a `Detector` and generating PPC config files

A `Detector` wraps a list of modules. When any module has `module_type != -1`, the detector is in **nextgen mode** and `ppc_sim` will write `om.conf` and `om.map` before calling PPC.

### 4.1 `om.conf` — module type definitions

One entry per unique `module_type` in the detector. If the same type ID appears on multiple modules (e.g. every string has DEggs), it is written only once using the first module encountered.

In [ ]:
import tempfile, os

# Build a small string of 5 DEgg modules
degg_string = [
    Module(
        pos=np.array([0.0, 0.0, float(i) * 17.0]),
        key=(1, i + 1),
        module_type=120,
        Rr=0.150, Rz=0.267,
        beta=0.5,
        n_pmts=2,
        pmt_dirs=[(180.0, 0.0), (0.0, 0.0)],
    )
    for i in range(5)
]

det = Detector(degg_string, Medium.ICE)

print("needs_nextgen():", det.needs_nextgen())  # True

tmpdir = tempfile.mkdtemp()
om_conf_path = os.path.join(tmpdir, "om.conf")
det.to_om_conf(om_conf_path)

print("\nom.conf contents:")
with open(om_conf_path) as f:
    print(f.read())

**om.conf column layout:**
```
name    type_id  area  beta  Rr     Rz     n_pmts  zen_0 az_0  [cable]
                                                    zen_1 az_1        ← continuation rows for PMTs 1…N
```
PPC reads this file once at startup and indexes module types by `type_id`.

### 4.2 `om.map` — per-DOM type assignment

Maps each physical DOM to its type ID. Legacy modules (`module_type=-1`) are omitted — PPC handles them via `as.dat`.

In [ ]:
om_map_path = os.path.join(tmpdir, "om.map")
det.to_om_map(om_map_path)

print("om.map contents (string_id  om_id  type_id):")
with open(om_map_path) as f:
    print(f.read())

### 4.3 Mixed detector: legacy + nextgen

A single detector can have a mix of legacy and nextgen modules. `needs_nextgen()` returns `True` as soon as any module has `module_type != -1`. Only the nextgen modules appear in `om.map`.

In [ ]:
mixed_modules = [
    Module(pos=np.array([0.0, 0.0, 0.0]),  key=(1, 1)),           # legacy DOM
    Module(pos=np.array([0.0, 0.0, 17.0]), key=(1, 2)),           # legacy DOM
    Module(pos=np.array([0.0, 0.0, 34.0]), key=(1, 3),
           module_type=120, Rr=0.150, Rz=0.267,                   # DEgg
           n_pmts=2, pmt_dirs=[(180.0, 0.0), (0.0, 0.0)]),
]
mixed_det = Detector(mixed_modules, Medium.ICE)
print("needs_nextgen():", mixed_det.needs_nextgen())

mixed_det.to_om_map(os.path.join(tmpdir, "mixed_om.map"))
print("\nom.map (only nextgen modules appear):")
with open(os.path.join(tmpdir, "mixed_om.map")) as f:
    print(f.read())

## 5. Running photon propagation and reading hits

The cell below requires a compiled PPC binary and ice tables. Set `PPC_EXE` and `PPC_TABLES_DIR` to your paths; skip to §6 if you don't have them.

**What happens under the hood when `ppc_sim` is called on a nextgen detector:**
1. `det.to_om_conf(ppc_tmpdir/om.conf)` — write type definitions
2. `det.to_om_map(ppc_tmpdir/om.map)` — write per-DOM type mapping
3. Copy `om.dirs` from the tables directory (PPC uses this for geometric normalization)
4. Set `PPCTABLESDIR=ppc_tmpdir` in the subprocess environment
5. Run PPC; it reads `om.conf` automatically when it finds it in `PPCTABLESDIR`
6. Parse the output; hits in nextgen mode have the form `HIT string dom_pmt time ...` where `dom_pmt` is e.g. `3_1` (om_id=3, pmt_id=1)

In [ ]:
import shutil
from pathlib import Path
from prometheus.lepton_propagation.loss import Loss
from prometheus.photon_propagation.ppc_photon_propagator import ppc_sim

PPC_EXE    = os.environ.get("PPC_EXE", "")
PPC_TABLES = os.environ.get("PPC_TABLES_DIR", "")
HAVE_PPC   = bool(PPC_EXE and os.path.isfile(PPC_EXE) and PPC_TABLES and os.path.isdir(PPC_TABLES))

if HAVE_PPC:
    sim_dir = Path(tempfile.mkdtemp()) / "sim"
    shutil.copytree(PPC_TABLES, str(sim_dir))

    cfg = {
        "paths": {
            "ppc_tmpdir": str(sim_dir),
            "ppc_tmpfile": "hits.tmp",
            "f2k_tmpfile": "losses.f2k",
            "ppctables": PPC_TABLES,
            "ppc_exe": PPC_EXE,
            "om_dirs": "",          # falls back to ppctables/om.dirs
        },
        "simulation": {"device": 0, "supress_output": False},
    }

    # Fake particle: charged pion at the centre, depositing 5 TeV
    class _Particle:
        pdg_code = 211
        e = 5000.0
        position = np.zeros(3)
        direction = np.array([0.0, 0.0, 1.0])
        children = []
        hits = []
        losses = [Loss(211, 5000.0, np.zeros(3))]
        def __int__(self):  return 211
        def __abs__(self):  return 211
        def __str__(self):  return "PiPlus"

    p = _Particle()
    ppc_sim(p, det, None, cfg)
    print(f"Total hits: {len(p.hits)}")
    print(f"PMT IDs seen: {sorted({h.pmt_id for h in p.hits})}")
else:
    print("PPC not available — set PPC_EXE and PPC_TABLES_DIR to run this cell.")
    print("Continuing with pre-built example hits.")

## 6. Understanding the `Hit` object

Each photon that reaches a module produces a `Hit` dataclass:

```python
@dataclass
class Hit:
    string_id:      int            # which string
    om_id:          int            # which DOM on that string
    time:           float          # photon arrival time [ns]
    wavelength:     Optional[float]  # photon wavelength [nm]
    om_zenith:      Optional[float]  # impact zenith on module surface [rad]
    om_azimuth:     Optional[float]  # impact azimuth on module surface [rad]
    photon_zenith:  Optional[float]  # photon travel direction zenith [rad]
    photon_azimuth: Optional[float]  # photon travel direction azimuth [rad]
    pmt_id:         Optional[int]    # PMT index within module (None = legacy)
```

The key difference between legacy and nextgen output is **`pmt_id`**:
- Legacy: `pmt_id = None` (only one PMT per module, not distinguished)
- Nextgen: `pmt_id = 0`, `1`, … up to `n_pmts - 1`

PPC outputs nextgen hits in the format:
```
HIT  string_id  om_id_pmtidx  time  wavelength  om_zen  om_az  ph_zen  ph_az
HIT  1          3_1           842.3  405.0       1.12    2.34   0.87    1.09
```
The `parse_ppc` function detects the underscore in token 2 and splits it into `om_id` and `pmt_id`.

In [ ]:
# Demonstrate parse_ppc directly on synthetic nextgen output
from prometheus.photon_propagation.utils.parse_ppc import parse_ppc

synthetic_output = """\
HIT 1 1_0 100.0 405.0 2.1 0.5 1.8 3.0
HIT 1 1_1 110.0 410.0 0.9 1.2 1.8 3.0
HIT 1 2_0 250.0 400.0 1.5 0.8 1.8 3.0
HIT 1 3_1 480.0 395.0 2.7 3.1 1.8 3.0
"""

import tempfile
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write(synthetic_output)
    fname = f.name

hits = parse_ppc(fname)
print(f"{'string':>8} {'om':>4} {'pmt':>4} {'time':>8} {'wl':>7}")
print("-" * 40)
for h in hits:
    print(f"{h.string_id:>8} {h.om_id:>4} {h.pmt_id:>4} {h.time:>8.1f} {h.wavelength:>7.1f}")

## 7. Serializing hits: the `output_mode` parameter

`serialize_particles_to_awkward` converts hits from all particles/events into an awkward array suitable for saving as Parquet. Three modes are supported:

| Mode | Fields included |
|------|-----------------|
| `"minimal"` | `sensor_pos_{x,y,z}`, `string_id`, `sensor_id`, `t`, `id_idx` |
| `"standard"` | minimal + `wavelength`, `pmt_id`, `om_zenith`, `om_azimuth`, `photon_zenith`, `photon_azimuth` |
| `"extended"` | standard + `hit_x`, `hit_y`, `hit_z` (Cartesian impact point in module-centred frame [m]) |

The `pmt_id` column is present in both `standard` and `extended` modes.

The `hit_{x,y,z}` fields in extended mode are computed from `om_zenith`, `om_azimuth`, and the module's `Rr`/`Rz` using spheroid or cylinder geometry, giving the physical position on the module surface where the photon was detected.

In [ ]:
import awkward as ak
from prometheus.photon_propagation.hit import Hit
from prometheus.utils.serialization.serialize_particles_to_awkward import serialize_particles_to_awkward

# Build a minimal injection-like structure to feed the serializer
example_hits = [
    Hit(1, 1, 100.0, 405.0, 2.1, 0.5, 1.8, 3.0, pmt_id=0),
    Hit(1, 1, 110.0, 410.0, 0.9, 1.2, 1.8, 3.0, pmt_id=1),
    Hit(1, 2, 250.0, 400.0, 1.5, 0.8, 1.8, 3.0, pmt_id=0),
    Hit(1, 3, 480.0, 395.0, 2.7, 3.1, 1.8, 3.0, pmt_id=1),
]

class _FakeParticle:
    serialization_idx = 0
    children = []
    def __init__(self, hits):
        self.hits = hits

class _FakeEvent:
    def __init__(self, hits):
        self.final_states = [_FakeParticle(hits)]

class _FakeInjection:
    def __iter__(self):
        return iter([_FakeEvent(example_hits)])

# The detector needs to know Rr/Rz to compute hit_x/y/z
class _FakeDetector:
    def __getitem__(self, key):
        return degg_string[key[1] - 1]   # reuse the DEgg modules built earlier

for mode in ("minimal", "standard", "extended"):
    arr = serialize_particles_to_awkward(_FakeDetector(), _FakeInjection(), output_mode=mode)
    print(f"\n--- output_mode='{mode}' ---")
    print("fields:", arr.fields)

## 8. Customising the setup

Here we look at the most common things you might want to change.

### 8.1 Changing PMT directions

The DEgg above has one up and one down PMT. You could tilt them, add a horizontal equatorial PMT, or point all PMTs in the same direction.

In [ ]:
# Tilted DEgg: PMTs at ±45° instead of ±90° from equator
tilted_degg = Module(
    pos=np.zeros(3), key=(1, 1),
    module_type=121,           # use a new type ID to avoid collision
    Rr=0.150, Rz=0.267, beta=0.5,
    n_pmts=2,
    pmt_dirs=[
        (135.0, 0.0),   # PMT 0: 45° below horizontal (zenith=135°)
        (45.0,  0.0),   # PMT 1: 45° above horizontal (zenith=45°)
    ],
)

# Visualise PMT directions on the spheroid
fig, ax = plt.subplots(figsize=(5, 5))
theta = np.linspace(0, 2 * np.pi, 300)
ax.plot(0.150 * np.cos(theta), 0.267 * np.sin(theta), "k-", lw=1.5, label="spheroid")

for i, (zen, az) in enumerate(tilted_degg.pmt_dirs):
    zen_rad = np.radians(zen)
    # Arrow from centre in PMT pointing direction
    dx = np.sin(zen_rad) * np.cos(np.radians(az))
    dz = np.cos(zen_rad)
    ax.annotate("", xy=(dx * 0.13, dz * 0.24), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=f"C{i}", lw=2))
    ax.text(dx * 0.16, dz * 0.26, f"PMT {i}\n{zen}°", color=f"C{i}", ha="center", fontsize=9)

ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("z [m]")
ax.set_title("Tilted DEgg PMT directions (xz-plane)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 8.2 Changing the aspect ratio

Increasing `Rz/Rr` makes the module more needle-like; decreasing it flattens it into a disk.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
theta = np.linspace(0, 2 * np.pi, 300)
Rr = 0.150

for ax, (Rz, label) in zip(axes, [
    (0.10, "Rz/Rr=0.67 (oblate)"),
    (0.15, "Rz/Rr=1.00 (sphere)"),
    (0.267, "Rz/Rr=1.78 (DEgg)"),
    (0.40,  "Rz/Rr=2.67 (elongated)"),
]):
    ax.plot(Rr * np.cos(theta), Rz * np.sin(theta), lw=2)
    ax.set_aspect("equal")
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("x [m]", fontsize=8)
    ax.set_ylabel("z [m]", fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Effect of Rz on spheroid shape (Rr=0.150 m fixed)", y=1.02)
plt.tight_layout()
plt.show()

### 8.3 Changing `beta` — angular acceptance

For nextgen modules the `beta` from `Module` is written to `om.conf` and applies to every PMT in that type. If two module types have different beta values (e.g. different PMT glass), give them different `module_type` IDs.

In [ ]:
# Two DEgg variants: standard beta vs isotropic
degg_standard = Module(
    pos=np.zeros(3), key=(1, 1), module_type=120,
    Rr=0.150, Rz=0.267, beta=0.5,
    n_pmts=2, pmt_dirs=[(180.0, 0.0), (0.0, 0.0)]
)

degg_isotropic = Module(
    pos=np.zeros(3), key=(1, 2), module_type=121,   # different type ID!
    Rr=0.150, Rz=0.267, beta=0.0,                  # flat angular response
    n_pmts=2, pmt_dirs=[(180.0, 0.0), (0.0, 0.0)]
)

det2 = Detector([degg_standard, degg_isotropic], Medium.ICE)
det2.to_om_conf(os.path.join(tmpdir, "two_type_om.conf"))

print("om.conf with two types (different beta):")
with open(os.path.join(tmpdir, "two_type_om.conf")) as f:
    print(f.read())

### 8.4 Adding a cable direction

PPC can account for the cable shadow (it blocks some photons). Set `cable_azimuth` to the cable attachment azimuth in the module frame [degrees]; it is appended as the last column of the `om.conf` row.

In [ ]:
degg_with_cable = Module(
    pos=np.zeros(3), key=(1, 1), module_type=120,
    Rr=0.150, Rz=0.267, beta=0.5,
    n_pmts=2, pmt_dirs=[(180.0, 0.0), (0.0, 0.0)],
    cable_azimuth=45.0,    # cable runs at 45° azimuth
)

det3 = Detector([degg_with_cable], Medium.ICE)
cable_conf = os.path.join(tmpdir, "cable_om.conf")
det3.to_om_conf(cable_conf)

print("om.conf with cable column:")
with open(cable_conf) as f:
    print(f.read())

### 8.5 The `area` parameter — global efficiency scaling

`area` multiplies the photon-detection probability for the entire module type. It is distinct from `efficiency` (which is used by Prometheus internally for noise and threshold calculations). Use `area` to:
- Scale a type to match calibration data without changing other parameters
- Quickly study systematic uncertainty: vary area ± 10 % and re-run

`area=1.0` means "use the efficiency encoded in `om.wv_{type}.{subtype}`" with no additional scaling.

## 9. What is and is not yet supported

### Supported
- Arbitrary number of PMTs per module (`n_pmts` + `pmt_dirs`)
- Spherical, spheroidal, and cylindrical module shapes (`Rr`, `Rz`)
- Per-type angular acceptance (`beta`)
- Per-type efficiency scaling (`area`)
- Cable shadow direction (`cable_azimuth`)
- Mixed detectors: legacy and nextgen modules on the same string or in the same run
- Hit-level PMT identification (`pmt_id` in Hit and in Parquet output)
- Cartesian hit position on module surface (`hit_x/y/z` in extended output mode)

### Not yet supported / caveats
- **Per-PMT QE spectra:** `om.wv_{type}.{subtype}` defines one wavelength-dependent QE curve per *subtype* (integer), but all PMTs of the same type share it. Distinct QE curves per individual PMT require separate `module_type` IDs.
- **PMT-level noise and dark rate:** `noise_rate` and `efficiency` in `Module` apply at the module level. There is no per-PMT noise differentiation.
- **Optical coupling / gel layers:** Not modelled; Prometheus delegates all photon transport to PPC.
- **Hit position uncertainty:** The `hit_x/y/z` extended output assumes a perfect spheroid or cylinder; it does not account for glass thickness or wavelength-dependent refraction.
- **`om_id = 0`:** PPC's `isinice()` gate excludes this value. Always start `key=(string, 1)` or higher.

## 10. Quick reference: building a DEgg string end-to-end

In [ ]:
import numpy as np
from prometheus.detector.module import Module
from prometheus.detector.detector import Detector
from prometheus.detector.medium import Medium

N_DOMS = 10
DOM_SPACING = 17.0   # metres

modules = [
    Module(
        pos=np.array([0.0, 0.0, i * DOM_SPACING]),
        key=(1, i + 1),          # om_id starts at 1
        module_type=120,          # DEgg type
        Rr=0.150,
        Rz=0.267,
        beta=0.5,
        area=1.0,
        n_pmts=2,
        pmt_dirs=[(180.0, 0.0),   # PMT 0: down
                  (0.0,   0.0)],  # PMT 1: up
    )
    for i in range(N_DOMS)
]

det = Detector(modules, Medium.ICE)
print(f"Detector: {det.n_modules} modules, needs_nextgen={det.needs_nextgen()}")
print(f"Extent z: {det.module_coords[:, 2].min():.0f} to {det.module_coords[:, 2].max():.0f} m")

# When passed to ppc_sim, this will automatically produce om.conf + om.map
# and parse hit lines like "HIT 1 4_1 ..." giving pmt_id=1 for the upward PMT.